# Chapter 1 — Colab runner

**Runtime → Change runtime type → T4 GPU.**

Colab differs from Kaggle in three ways that matter, and every one of them has
bitten this project already:

| | Kaggle | Colab |
|---|---|---|
| GPUs | 2 × T4 — jobs could be paired | **1** — everything is sequential |
| Disk | survives the session | **wiped on disconnect** → we mount Drive |
| Session | long | **drops on idle / browser close** → every cell is resume-safe |

**Re-running any cell is safe.** Training skips a condition whose adapter already
exists on Drive, so after a disconnect you re-run the same cell and it continues
from where it stopped.

---
### ⚠️ Before anything else
If your Kaggle session still exists, download `chapter1_results.zip` and the
`checkpoints/` folder from it and drop them in Drive at
`MyDrive/kgc-thesis/`. Otherwise the adapters already trained (YAGO3-10 A B C G,
WN11 A B) have to be retrained — about 2.7 GPU-hours.


## 0 · Drive, repo, environment


In [ ]:
# ── Drive is the ONLY durable storage on Colab ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json, shutil
from pathlib import Path

PERSIST = Path('/content/drive/MyDrive/kgc-thesis')   # survives disconnects
REPO    = Path('/content/repo')                       # wiped every session
REPO_URL = 'https://github.com/lynda-lagh/contribution-.git'

for sub in ('checkpoints', 'results', 'data'):
    (PERSIST / sub).mkdir(parents=True, exist_ok=True)
print('persistent store:', PERSIST)
for sub in ('checkpoints', 'results', 'data'):
    n = sum(1 for _ in (PERSIST / sub).rglob('*'))
    print(f'  {sub:12s} {n:5d} files already there')


In [ ]:
# ── clone or update, then LINK the heavy dirs onto Drive ────────────────────
def sh(*c, check=True):
    r = subprocess.run(c, capture_output=True, text=True)
    if check and r.returncode:
        raise SystemExit(f"$ {' '.join(c)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

before = sh('git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD', check=False) or '(none)'
if (REPO / '.git').exists():
    sh('git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', 'main')
    sh('git', '-C', str(REPO), 'reset', '--hard', 'FETCH_HEAD')
else:
    sh('git', 'clone', '--depth', '1', REPO_URL, str(REPO))
print('repo @', sh('git', '-C', str(REPO), 'log', '-1', '--pretty=%h  %s'))
print('moved', before, '->', sh('git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'))

os.chdir(REPO)
# data/ checkpoints/ results/ live on Drive; the repo just points at them.
# ⚠️ symlink, not copy — a 3 GB checkpoint copied per session would be absurd.
for sub in ('checkpoints', 'results', 'data'):
    link = REPO / sub
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(PERSIST / sub)
    print(f'  {sub:12s} -> {PERSIST/sub}')

sys.path.insert(0, str(REPO))


In [ ]:
# ── the pinned stack. Colab ships newer transformers than this code wants ───
PIN = '4.57.6'
import importlib

def stack():
    try:
        import peft, transformers
        importlib.reload(peft); importlib.reload(transformers)
        return peft.__version__, transformers.__version__
    except Exception:
        return None, None

p, t = stack()
if t != PIN:
    print(f'installing… (have peft={p} transformers={t})')
    !pip install -q "transformers=={PIN}" peft accelerate datasets bitsandbytes
    !pip install -q pyyaml tqdm scikit-learn
!pip uninstall -y -q torchao 2>/dev/null

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu  ', torch.cuda.get_device_name(0))
    print('vram ', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GB')
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU.')
print('\n⚠️ if transformers was just installed, Runtime -> Restart session, then',
      'run cells 1-3 again (Drive + repo + this one). Nothing is lost.')


In [ ]:
DATASET = 'YAGO3-10'
LIMIT   = 2000          # test items per arm, identical across conditions
!python -m chapter1.test_chapter1


## 1 · Data

Skips anything already on Drive. The negatives are **type-consistent**, not
random — that is what removed the type-tag leak (62.4% → 51.3%). Do not change
it back without re-running `check_type_leak`.


In [ ]:
from pathlib import Path
import subprocess, sys

def sh_show(cmd):
    print('$', cmd, flush=True)
    return subprocess.call(cmd, shell=True)

if not Path('data/YAGO3-10/train.tsv').exists():
    sh_show('python -m scripts.fetch_data --datasets YAGO3-10 WN11')
else:
    print('[skip] raw datasets already on Drive')

# YAGO3-10 ships no ±1 test labels. --regenerate rewinds to test.original.tsv
# first, so this is safe to re-run and can never destroy shipped labels.
probe = subprocess.run([sys.executable, '-c',
    "import sys; sys.path.insert(0,'.');"
    "from src.data.loaders import load_kg;"
    "kg=load_kg('YAGO3-10','data');"
    "print(sum(1 for t in kg.test if t.label is not None))"],
    capture_output=True, text=True)
labelled = int((probe.stdout or '0').strip() or 0)
if labelled:
    print(f'[skip] YAGO3-10 test already has {labelled:,} labels')
else:
    sh_show('python -m scripts.make_test_negatives --dataset YAGO3-10 '
            '--strategy type_consistent --seed 42')


In [ ]:
!python -m chapter1.validate --dataset YAGO3-10 WN11
!python -m chapter1.data --all --dataset YAGO3-10
!python -m chapter1.check_type_leak --dataset YAGO3-10 --condition C D E G --json results/type_leak_YAGO3-10.json


### Changing the test negatives (only if you mean to)

The cell above **skips** if `test.tsv` already carries labels, so re-running it
never silently rebuilds your test set. To actually change the strategy you have
to say so — `--regenerate` rewinds to `test.original.tsv` first, so the guard can
only ever wind back to the untouched original and can never destroy shipped
labels like WN11's.

⚠️ **This invalidates every result already computed.** The test set changes, so
all accuracies, the type-tag floor and the closed-world rejection count have to
be recomputed. Only do it deliberately.


In [ ]:
# UNCOMMENT DELIBERATELY. Changes the test set; invalidates existing results.
#
# type-consistent (current):  tag-only floor 51.3%, 690 candidates rejected
# random (the original):      tag-only floor 62.4%  <- the leak, do not go back
#
# !python -m scripts.make_test_negatives --dataset YAGO3-10 --strategy type_consistent --seed 42 --regenerate
# !python -m chapter1.validate --dataset YAGO3-10
# !python -m chapter1.data --all --dataset YAGO3-10
# !python -m chapter1.check_type_leak --dataset YAGO3-10 --condition C D E G
print('nothing run — uncomment above if you really want to rebuild the test set')


In [ ]:
# Understand the graph before designing on it. Free, no GPU.
# Ends with a verdict on which conditions are viable for this dataset.
!python -m chapter1.profile_data --dataset YAGO3-10 --json results/profile.json


## 2 · Train and evaluate — one GPU, resume-safe

Order matters and it is not the order the conditions are named in:

| | condition | why here |
|---|---|---|
| 1 | **A, B** | the decomposition — the headline. Nothing else means anything without it |
| 2 | **S** | shuffled names. The one control that can invalidate the headline |
| 3 | **C, G** | the type question, read against the 51.3% floor |
| 4 | **D** | negative hardness |
| 5 | **E** | negative count — 70k instances, ~2 h alone |

**A condition whose adapter is already on Drive is skipped.** After a disconnect,
re-run this cell.


In [ ]:
import time
from pathlib import Path

ORDER = ['A', 'B', 'S', 'C', 'G', 'D', 'E']

def adapter_of(c, ds=None):
    return Path('checkpoints') / f'ch1-{ds or DATASET}-{c}'

def trained(c, ds=None):
    return (adapter_of(c, ds) / 'adapter_config.json').exists()

def status():
    print(f"{'cond':6s} {'trained':>8s}   adapter")
    for c in ORDER:
        print(f'{c:6s} {str(trained(c)):>8s}   {adapter_of(c)}')

status()


In [ ]:
# ── the runner. Re-run freely; finished conditions are skipped. ─────────────
def go(conds, dataset=None, train=True, evaluate=True, extra=''):
    ds = dataset or DATASET
    for c in conds:
        do_train = train and not trained(c, ds)
        if train and not do_train:
            print(f'\n[skip] {c}: adapter already on Drive')
        flags = ('--train ' if do_train else '') + ('--evaluate ' if evaluate else '')
        if not flags.strip():
            continue
        cmd = (f'python -m chapter1.run --dataset {ds} --condition {c} '
               f'{flags} --limit {LIMIT} {extra}')
        print(f'\n{"="*70}\n{c}   {cmd}\n{"="*70}', flush=True)
        t0 = time.time()
        rc = subprocess.call(cmd, shell=True)
        print(f'[{c}] rc={rc}  {(time.time()-t0)/60:.1f} min', flush=True)
        if rc:
            print(f'✋ {c} failed — stopping so the failure is not buried.')
            break

go(['A', 'B'])          # ~40 min each


In [ ]:
go(['S'])       # ★ the control that defends the chapter


In [ ]:
go(['C', 'G'])  # the type question


In [ ]:
go(['D'])       # negative hardness


In [ ]:
# E is 70,000 instances — roughly 2 h on one T4.
# Keep the tab awake, or accept a disconnect and re-run: it resumes by skipping.
go(['E'])


### Re-evaluate without retraining

Training and evaluation happen in one call, so normally you never need this.
You need it when **training succeeded but scoring failed** — which has happened
twice: once on a missing `data/*-anon/built/` path, once on `float()` applied to
a dict inside the calibration helper. Both times the adapter was fine and the
numbers were lost.

This finds every trained condition with no result file and scores only those.
~4 min per condition. It never retrains.


In [ ]:
import glob

def has_result(cond, ds=None):
    ds = ds or DATASET
    return bool(glob.glob(f'results/*{ds}-{cond}-eval*.json'))

missing = [c for c in ORDER if trained(c) and not has_result(c)]
print('trained but unscored:', missing or 'none — nothing to do')

# go() with train=False evaluates only; nothing is retrained.
if missing:
    go(missing, train=False)

# force a re-score:  go(['A','B'], train=False)


## 3 · Collect — read, do not recompute

`--train --evaluate` already produced every number. This reads them off disk.


In [ ]:
import glob, json
rows = []
for f in sorted(glob.glob('results/*eval*.json')):
    d = json.load(open(f))
    rows.append((Path(f).stem, d.get('acc_real'), d.get('acc_anon'), d.get('gap')))
print(f"{'run':42s} {'real':>8s} {'anon':>8s} {'gap':>8s}")
for r in rows:
    fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
    print(f'{r[0]:42s} {fmt(r[1]):>8s} {fmt(r[2]):>8s} {fmt(r[3]):>8s}')
print(f'\n{len(rows)} runs found. Missing ones failed — re-run just those.')


In [ ]:
# ★ THE HEADLINE. Matched arms: each adapter on the data it was trained for.
#   NOT the acc_real/acc_anon pair inside a single run — that one mixes in
#   distribution shift, because A never saw an `entityN` during training.
def cell(cond, key):
    for f in glob.glob(f'results/*{DATASET}-{cond}-eval*.json'):
        return json.load(open(f)).get(key)

real, anon = cell('A', 'acc_real'), cell('B', 'acc_anon')
if real and anon:
    above = real - 0.5
    mem   = real - anon
    print(f'  tuned  (A on real)      {real:.4f}')
    print(f'  anon   (B on anon)      {anon:.4f}')
    print(f'  above chance            {above:.4f}')
    print(f'  memorisation            {mem:.4f}')
    print(f'  residual knowledge      {above - mem:.4f}')
    print(f'  MEMORISATION SHARE      {mem/above:.1%}')
else:
    print('A and/or B not evaluated yet')


## 4 · Analysis, report, ranking


In [ ]:
!python -m chapter1.analysis --dataset {DATASET}
!python -m chapter1.report   --dataset {DATASET}


In [ ]:
# 50-way filtered link prediction — turns the classifier into a ranker (MRR).
for c in ['A', 'B', 'C', 'S']:
    if not trained(c):
        print(f'[skip] {c}: not trained')
        continue
    cmd = (f'python -m chapter1.rank --adapter checkpoints/ch1-{DATASET}-{c} '
           f'--dataset {DATASET} --condition {c} --limit 500')
    print('$', cmd, flush=True)
    subprocess.call(cmd, shell=True)


In [ ]:
# SMI — slow, and only A and B carry the comparison against FLAME.
go(['A', 'B'], train=False, extra='--smi')


## 5 · WN11 — the second dataset

Already complete if you carried the Kaggle checkpoints across: untuned 0.6920,
tuned 0.9265, anonymised 0.5325 → **92.4%** memorisation share, balanced
familiarity gap **+0.0036**.

Re-run only if those adapters were lost.


In [ ]:
# ── WN11 diagnostics: the same free checks YAGO3-10 gets ───────────────────
# ⚠️ TYPE_TAG_FLOOR['WN11'] is still None, so floor_for() falls back to 0.5 for
#    WN11's typed conditions. On YAGO3-10 that assumption was wrong by 12.4
#    points. Measure it before comparing typed results across the two datasets.
!python -m chapter1.validate --dataset WN11
!python -m chapter1.profile_data --dataset WN11 --json results/profile_WN11.json
!python -m chapter1.data --all --dataset WN11
!python -m chapter1.check_type_leak --dataset WN11 --condition C D E G --json results/type_leak_WN11.json


In [ ]:
!python -m chapter1.data --condition A B S --dataset WN11
go(['A', 'B'], dataset='WN11')
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch1_diagnostic.analyse --dataset WN11
!python -m chapter1.seen_unseen --dataset WN11


### Export the fine-tuned adapters

LoRA trained **1,089,536 of 1,544,803,840** parameters — 0.07%. So the
fine-tuned artefact is ~**4 MB**, not 3 GB: the base model is untouched and is
re-downloaded on load.

The folders are big for two reasons that do not need keeping:

| | why it is there | keep? |
|---|---|---|
| `checkpoint-250/`, `checkpoint-500/` | optimiser + scheduler + RNG state, written every 250 steps | ❌ dead once the best model is saved |
| `tokenizer.json`, `vocab.json` | ~10 MB, **byte-identical** in every adapter | ❌ the base model id is recorded instead |
| `adapter_model.safetensors` | **the fine-tuning** | ✅ |

`--prune` deletes the first; the export drops the second. About **30 MB → 4 MB**
per adapter. `MANIFEST.json` records base model, LoRA hyper-parameters, runtime,
peak VRAM, the fit verdict and the accuracies. `USAGE.md` has the six lines that
load one.


In [ ]:
# Export straight onto Drive so it survives the session.
!python -m scripts.export_adapters --out /content/drive/MyDrive/kgc-thesis/export/adapters --zip --prune

!ls -la /content/drive/MyDrive/kgc-thesis/export/ 2>/dev/null
print('\n★ adapters.zip on Drive is every fine-tuned adapter, a few MB.')
print('  Reload without retraining — see export/adapters/USAGE.md.')


## 6 · Backup


In [ ]:
# results/ and checkpoints/ are ALREADY on Drive via the symlinks, so this is
# just a dated snapshot you can download in one click.
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
out = f'/content/drive/MyDrive/kgc-thesis/chapter1_results_{stamp}.zip'
!zip -qr "{out}" results/ data/*/built/manifest.json
print('wrote', out)
!du -sh /content/drive/MyDrive/kgc-thesis/checkpoints /content/drive/MyDrive/kgc-thesis/results
